In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# # Importing Libraries
# import re
# #import config
# import pandas as pd
# import numpy as np
# from pyomo.gdp import *
# from pyomo.environ import *
# from functools import reduce

%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import numpy as np
import re
from functools import reduce

import shutil
import sys
import os.path
import warnings
warnings.filterwarnings('ignore', '.*do not.*', )

pd.set_option('display.max_columns', 10000)
pd.set_option('display.max_rows', 10000)
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

assert(shutil.which("cbc") or os.path.isfile("cbc"))

from pyomo.environ import *
from pyomo.gdp import *

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 65.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 KB 4.9 MB/s eta 0:00:00
Selecting previously unselected package coinor-libcoinutils3v5.
(Reading database ... 128275 files and directories currently installed.)
Preparing to unpack .../0-coinor-libcoinutils3v5_2.11.4+repack1-1_amd64.deb ...
Unpacking coinor-libcoinutils3v5 (2.11.4+repack1-1) ...
Selecting previously unselected package coinor-libosi1v5.
Preparing to unpack .../1-coinor-libosi1v5_0.108.6+repack1-1_amd64.deb ...
Unpacking coinor-libosi1v5 (0.108.6+repack1-1) ...
Selecting previously unselected package coinor-libclp1.
Preparing to unpack .../2-coinor-libclp1_1.17.5+repack1-1_amd64.deb ...
Unpacking coinor-libclp1 (1.17.5+repack1-1) ...
Selecting previously unselected package coinor-libcgl1.
Preparing to unpack .../3-coinor-libcgl1_0.60.3+repack1-2_amd64.deb ...
Unpacking coinor-libcgl1 (0.60.3+repack1-2) ...
Selecting previously unselecte

In [ ]:
def pyomo_function(scene_summary_path, scene_path, CALENDAR_path, character_path, location_path):

    # Model type
    model = ConcreteModel()

    # Input data
    scene_summary = pd.read_csv(scene_summary_path)
    scene_summary.columns = ["Shoot_Location","Scene_Number","Shoot_LocationType","Day_Part", "Actors"]
    scene = pd.read_csv(scene_path)
    scene.columns = ["Scene_Number","Shoot_Town", "Shoot_time"]
    SCENE_SUMMARY = pd.merge(scene_summary,scene,on='Scene_Number',how='left')
    SCENE_SUMMARY["Loc_Town"] = SCENE_SUMMARY[["Shoot_Town", "Shoot_Location"]].apply("_".join, axis=1)
    CALENDAR = pd.read_csv(CALENDAR_path)
    CALENDAR.columns = ["DATE","D_ID"]
    CALENDAR['DATE'] = pd.to_datetime(CALENDAR['DATE'], format='%d-%m-%Y')
    character = pd.read_csv(character_path)
    character['from_date'] = pd.to_datetime(character['from_date'], format='%d-%m-%Y')
    character['to_date'] = pd.to_datetime(character['to_date'], format='%d-%m-%Y')
    ACTOR_SUMMARY = character[["characters","criticality"]]
    ACTOR_SUMMARY.columns = ["Actors", "Criticality"]
    df = character[["characters","from_date","to_date"]]
    df['DATE'] = [pd.date_range(s, e, freq='d') for s, e in
                  zip(df['from_date'],df['to_date'])]
    df = df.explode('DATE').drop(['from_date', 'to_date'], axis=1)
    df.columns = ["Actors","DATE"]
    ACTOR_AVAILABILITY =  pd.merge(df,CALENDAR,on='DATE',how='inner')
    location = pd.read_csv(location_path)
    location['from_date'] = pd.to_datetime(location['from_date'], format='%d-%m-%Y')
    location['to_date'] = pd.to_datetime(location['to_date'], format='%d-%m-%Y')
    LOCATION_SUMMARY = location[["location","criticality"]]
    LOCATION_SUMMARY.columns = ["Shoot_Location", "Criticality"]
    LOCATION_SUMMARY = pd.merge(SCENE_SUMMARY,LOCATION_SUMMARY,on='Shoot_Location',how='inner')
    LOCATION_SUMMARY.drop(["Shoot_LocationType","Day_Part","Actors","Shoot_time"],axis=1, inplace=True)
    df1 = location[["location",	"from_date", "to_date"]]
    df1['DATE'] = [pd.date_range(s, e, freq='d') for s, e in
                  zip(df1['from_date'],df1['to_date'])]
    df1 = df1.explode('DATE').drop(['from_date', 'to_date'], axis=1)
    df1.columns = ["Shoot_Location","DATE"]
    df1 = pd.merge(SCENE_SUMMARY,df1,on='Shoot_Location',how='inner')
    df1.drop(["Shoot_LocationType","Day_Part","Actors","Shoot_time"],axis=1, inplace=True)

    LOCATION_AVAILABILITY =  pd.merge(df1,CALENDAR,on='DATE',how='inner')
    LOCATION_AVAILABILITY
    ac_sc = dict()
    for idx, val in enumerate(SCENE_SUMMARY.Actors):
        ele = SCENE_SUMMARY.Scene_Number[idx]
        if pd.notna(val):
            val = re.sub("[^A-Za-z0-9, ]+", "", val).split(', ')
            ac_sc.setdefault(ele, []).append(val)
        else:
            ac_sc.setdefault(ele, []).append("")

    character_list = character["characters"].unique().tolist()
    sdf = pd.DataFrame()
    for i in ac_sc:
        if i in range(1, len(ac_sc)+1):
            for j in range(len(ac_sc[i][0])):
                sdf = sdf.append({ac_sc[i][0][j]:i}, ignore_index=True)
    sam_dict1 = {}
    for i in character_list:
        sam_dict1[i] = sdf[i].dropna().tolist()
    sdf1 = pd.DataFrame(sam_dict1.items(), columns=['Actors', 'Scenes'])
    data1 = [(row.Actors, sample) for row in sdf1.itertuples() for sample in row.Scenes]
    ACTOR_SCENES = pd.DataFrame(data1, columns=['Actors', 'Scene_Number'])
    ACTOR_INFO = pd.merge(ACTOR_AVAILABILITY,ACTOR_SCENES,on='Actors',how='inner')
    ACTOR_INFO.rename(columns= {'D_ID':'Availability'}, inplace=True)
    ACTOR_INFO['scen_avai'] = ACTOR_INFO[['Scene_Number','Availability']].apply(tuple, axis=1)
    ACTOR_INFO['actor_avai'] = ACTOR_INFO[['Actors','Availability']].apply(tuple, axis=1)
    ACTOR_INFO['scene_actor'] = ACTOR_INFO[['Scene_Number','Actors']].apply(tuple, axis=1)
    ACTOR_INFO['asn'] = ACTOR_INFO[['Actors','Scene_Number','Availability']].apply(tuple, axis=1)
    loc_list = SCENE_SUMMARY["Loc_Town"].unique().tolist()
    loc_sc = dict(zip(SCENE_SUMMARY.Scene_Number, SCENE_SUMMARY.Loc_Town))
    sdf2 = pd.DataFrame()

    for i in loc_sc:
        if i in range(1, len(loc_sc)+1):
            sdf2 = sdf2.append({loc_sc[i]:i}, ignore_index=True)

    sam_dict2 = {}
    for i in loc_list:
        sam_dict2[i] = sdf2[i].dropna().tolist()
    sdf3 = pd.DataFrame(sam_dict2.items(), columns=['Loc', 'Scenes'])
    data2 = [(row.Loc, sample) for row in sdf3.itertuples() for sample in row.Scenes]
    LOCATION_SCENES = pd.DataFrame(data2, columns=['Loc', 'Scene_Number'])
    split = LOCATION_SCENES['Loc'].str.split('_', 1, expand=True)
    LOCATION_SCENES = LOCATION_SCENES.assign(Shoot_Town=split[0], Shoot_Location=split[1])
    LOCATION_SCENES.drop('Loc', 1, inplace=True)
    LOCATION_INFO = pd.merge(LOCATION_AVAILABILITY,LOCATION_SCENES,on=['Shoot_Location','Shoot_Town','Scene_Number'],how='inner')
    LOCATION_INFO.rename(columns= {'D_ID':'Availability'}, inplace=True)
    LOCATION_INFO['scen_avai'] = LOCATION_INFO[['Scene_Number','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['town_avai'] = LOCATION_INFO[['Shoot_Town','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['loc_avai'] = LOCATION_INFO[['Shoot_Location','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['lsn'] = LOCATION_INFO[['Shoot_Location','Scene_Number','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['tsn'] = LOCATION_INFO[['Shoot_Town','Scene_Number','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['ij'] = LOCATION_INFO[['Shoot_Town','Shoot_Location']].apply(tuple, axis=1)
    LOCATION_INFO['sj'] = LOCATION_INFO[['Scene_Number','Shoot_Location']].apply(tuple, axis=1)

    scene_list = ACTOR_INFO.Scene_Number.unique().tolist()
    scene_list.sort()
    df1 = ACTOR_INFO.groupby(by=["Scene_Number"])
    #main_df = pd.DataFrame(["scen_avail"])
    main_list = []
    for i in scene_list:
        df2 = df1.get_group(i)
        actors_list = df2.Actors.unique().tolist()
        #print("---------------")

        list = []
        df3 = df2.groupby("Actors")
        for j in actors_list:
            list.append(df3.get_group(j))

        final_df = reduce(lambda  left,right: pd.merge(left,right,on=['Availability'], how='inner'), list)
        #print(final_df.shape)

        main_list.append(final_df.iloc[:,4:5].values)

    list1 =[]
    for i in range(len(main_list)):
      #print(i)
      list1.append(main_list[i].tolist())
    flat_list = [item for sublist in list1 for item in sublist]
    flat_list1 = [item for sublist in flat_list for item in sublist]
    Actor_avail = flat_list1
    location_avail = LOCATION_INFO.scen_avai.unique().tolist()
    def Intersection(lst1, lst2):
        return set(lst2).intersection(lst1)

    fin_lis = Intersection(Actor_avail,location_avail)
    FINAL_ACTOR_INFO = ACTOR_INFO.copy()
    FINAL_ACTOR_INFO = FINAL_ACTOR_INFO[FINAL_ACTOR_INFO['scen_avai'].isin(fin_lis)]
    FINAL_LOCATION_INFO = LOCATION_INFO.copy()
    FINAL_LOCATION_INFO = FINAL_LOCATION_INFO[FINAL_LOCATION_INFO['scen_avai'].isin(fin_lis)]

    # sets
    model.i = Set(initialize=LOCATION_SUMMARY["Shoot_Town"].tolist())
    model.j = Set(initialize=LOCATION_SUMMARY["Shoot_Location"].tolist())
    model.s = Set(initialize=SCENE_SUMMARY["Scene_Number"].tolist())
    model.a = Set(initialize=ACTOR_SUMMARY["Actors"].tolist())
    model.n = Set(initialize=CALENDAR['D_ID'].tolist())
    model.an = Set(initialize=FINAL_ACTOR_INFO["actor_avai"].tolist())
    model.sa = Set(initialize=FINAL_ACTOR_INFO["scene_actor"].tolist())
    model.asn = Set(initialize=FINAL_ACTOR_INFO["asn"].tolist())
    model.jn = Set(initialize=FINAL_LOCATION_INFO["loc_avai"].tolist())
    model.jsn = Set(initialize=FINAL_LOCATION_INFO["lsn"].tolist())
    model.tn = Set(initialize=FINAL_LOCATION_INFO["town_avai"].tolist())
    model.isn = Set(initialize=FINAL_LOCATION_INFO["tsn"].tolist())
    model.ij = Set(initialize=FINAL_LOCATION_INFO["ij"].tolist())
    model.sj = Set(initialize=FINAL_LOCATION_INFO["sj"].tolist())

    # parameters
    set_ele1 = pd.Series(ACTOR_SUMMARY.Criticality.values,index=ACTOR_SUMMARY.Actors).to_dict()
    model.CA = Param(model.a,initialize=set_ele1)
    set_elem = pd.Series(SCENE_SUMMARY.Shoot_time.values,index=SCENE_SUMMARY.Scene_Number).to_dict()
    model.TS = Param(model.s,initialize=set_elem)
    model.Can = Param(model.an, initialize=1)
    set_elex = pd.Series(LOCATION_SUMMARY.Criticality.values,index=LOCATION_SUMMARY.Shoot_Location).to_dict()
    model.CL = Param(model.j,initialize=set_elex)
    model.Bsa = Param(model.sa, initialize=1)
    model.Can = Param(model.an, initialize=1)
    model.Asj = Param(model.sj, initialize=1)
    model.Djn = Param(model.jn, initialize=1)
    model.Eij = Param(model.ij, initialize=1)
    set_ele4 = pd.Series(SCENE_SUMMARY.Shoot_time.values,index=SCENE_SUMMARY.Scene_Number).to_dict()
    model.Tsh = Param(model.s ,initialize=set_ele4)

    # variables
    model.Wjn = Var(model.jn, within=Binary)
    model.Dursn = Var( model.s,model.n)
    model.AVsn = Var( model.s,model.n)
    model.SHsn = Var( model.s,model.n)
    model.DELsn =Var(model.s,model.n, within=Binary)
    model.Usn = Var(fin_lis,within=Binary)
    model.Vasn = Var(model.asn, within=Binary)
    model.Win = Var(model.tn,within=Binary)
    model.Xjsn = Var( model.jsn ,within=Binary)

    # Constraints
    # Summation over n across Usn ==1
    def rule2(model, s):
        return sum(model.Usn[s,iter] for iter in model.n if (s!=None and (s,iter)) in model.Usn) == 1
    model.const1 = Constraint( model.s, rule=rule2 )

    # Summation over s Usn*TS(s) <= WT
    sc_num = []
    for iter1 in model.s:
        for iter2 in model.n:
            if (iter1,iter2) in model.Usn:
                sc_num.append(iter2)

    def rule10(model,n):
            return sum(model.Usn[iter,n]*model.TS[iter] for iter in model.s if (iter,n) in model.Usn) <= 8
    model.const2 = Constraint(sc_num, rule = rule10)

    #Vasn == Usn*Bsa*Cna
    model.const3 = ConstraintList()
    for iter1 in model.n:
        for iter2 in model.s:
            for iter3 in model.a:
                if (iter3,iter2,iter1) in model.Vasn and (iter2,iter1) in model.Usn and (iter3,iter1) in model.Can and (iter2,iter3) in model.Bsa:
                    model.const3.add(model.Vasn[iter3,iter2,iter1] == model.Usn[iter2,iter1] * model.Bsa[iter2,iter3] * model.Can[iter3,iter1])
                else:
                    continue

    #Xjsn == Usn * Asj *Djn
    model.const4 = ConstraintList()
    for iter1 in model.j:
        for iter2 in model.n:
            for iter3 in model.s:
                if (iter1,iter3,iter2) in model.Xjsn and (iter1,iter2) in model.Djn and (iter3,iter1) in model.Asj:
                    model.const4.add(model.Xjsn[iter1,iter3,iter2] == model.Usn[iter3,iter2]*model.Asj[iter3,iter1]*model.Djn[iter1,iter2])
                else:
                    continue

    #Xjsn <= Win * Eij * Djn
    model.const5 = ConstraintList()
    for iter1 in model.j:
        for iter2 in model.s:
             for iter3 in model.n:
                for iter4 in model.i:
                    if (iter1,iter2,iter3) in model.Xjsn and (iter4,iter3) in model.Win and (iter1,iter3) in model.Djn and (iter4,iter1) in model.Eij:
                        model.const5.add(model.Xjsn[iter1,iter2,iter3] <= model.Win[iter4,iter3] * model.Eij[iter4,iter1]* model.Djn[iter1,iter3])
                    else:
                        continue
    # Summation over i across Win <=1
    sc_num1 = []
    for iter in model.tn:
        if (iter) in model.tn:
            sc_num1.append(iter[1])


    def rule4(model, n):
        return sum(model.Win[iter1,n] for iter1 in model.i if(iter1,n) in model.tn) <= 1
    model.const6 = Constraint(sc_num1, rule=rule4)

    model.const7 = ConstraintList()
    for iter1 in model.i:
        for iter3 in model.i:
            for iter2 in model.n:
                if iter2 != 1 and iter1!=iter3:
                    if (iter1,iter2) in model.Win and (iter3,iter2-1) in model.Win :
                        model.const7.add(model.Win[iter1,iter2]<= 1 - model.Win[iter3,iter2-1])
                else:
                    continue

    #objective
    model.obj1 = Objective( expr = sum( model.Xjsn[iter1]*(1/model.CL[iter1[0]]) for iter1 in model.jsn) + sum(model.Vasn[iter2]* (1/model.CA[iter2[0]]) for iter2 in model.asn))

    # Solver
    solver = SolverFactory('cbc')
    resolve = solver.solve(model)
    resolve.write()

    day_num1 = []
    loc = []
    scene_nums1 = []
    for x in model.jsn:
        if model.Xjsn[x]() == 1.0:
            day_num1.append(x[2])
            loc.append(x[0])
            scene_nums1.append(x[1])
    res_df2 =  pd.DataFrame(
        {'D_ID': day_num1,
         'Location': loc,
         'Scene_Num': scene_nums1
        })

    day_num = []
    actor = []
    scene_nums = []
    for x in model.asn:
        if model.Vasn[x]() == 1.0:
            day_num.append(x[2])
            actor.append(x[0])
            scene_nums.append(x[1])

    res_df1 =  pd.DataFrame(
        {'D_ID': day_num,
         'Actor': actor,
         'Scene_Num': scene_nums
        })
    numbers = []
    town = []
    for iter1 in model.tn:
        if model.Win[iter1]() == 1.0:
            numbers.append(iter1[1])
            town.append(iter1[0])

    res_df3 =  pd.DataFrame(
        {'D_ID': numbers,
         'Shoot_Town': town
        })
    res_df4 = pd.merge(res_df2,res_df3,on='D_ID',how='inner')
    res_df5 = pd.merge(res_df1,res_df4,on=['D_ID','Scene_Num'],how='inner')
    final_df = pd.merge(res_df5,CALENDAR,on='D_ID',how='inner')
    final_df = final_df.sort_values("D_ID")
    final_df = final_df.reindex(['Shoot_Town','Scene_Num','Location','Actor','DATE',"D_ID"], axis=1)
    final_df.groupby((final_df['Shoot_Town'].shift() != final_df['Shoot_Town']).cumsum())

    dfs = []
    for k, v in final_df.groupby((final_df['Shoot_Town'].shift() != final_df['Shoot_Town']).cumsum()):
        v["Schedule"] = k
        dfs.append(v)
    final_df1 = pd.concat(dfs)
    final_df1 = final_df1.reindex(["Schedule",'Shoot_Town','Scene_Num','Location','Actor','DATE',"D_ID"], axis=1)
    final_df2 = final_df1.groupby(['Schedule'])
    final_df2 = final_df2.agg(Minimum_Date=('DATE', np.min), Maximum_Date=('DATE', np.max))
    final_df2['Schedule'] = final_df2.index
    final_df2['list_of_tuples'] = final_df2[['Schedule', 'Minimum_Date','Maximum_Date']].apply(tuple, axis=1)
    list_of_tuples = final_df2['list_of_tuples'].to_list()
    From_date = []
    To_date = []
    schedule_list = final_df2['Schedule'].tolist()
    for i in range(len(schedule_list)):
        for row in final_df1['Schedule']:
            if row == i+1:  From_date.append(list_of_tuples[i][1]),To_date.append(list_of_tuples[i][2])

    final_df1['From_Date'] = From_date
    final_df1['To_Date'] = To_date
    gk = final_df1.groupby(["Schedule","Shoot_Town"])
    def getList(dict):
        list = []
        for key in dict.keys():
            list.append(key)

        return list

    gk_dict = gk.groups
    gk_group = getList(gk_dict)
    dfs = []
    for group in gk_group:
      group1 = gk.get_group(group)
      gk1 = group1.groupby('Scene_Num')
      gk1_group = getList(gk1.groups)
      for num in gk1_group:
        subgroup1_1 = gk1.get_group(num)
        df_new = subgroup1_1.groupby('Scene_Num', as_index=False).agg({'Actor' : ', '.join,'D_ID': 'first','Scene_Num':'first', 'Location':'first', 'Shoot_Town':'first', 'DATE':'first','Schedule':'first','To_Date':'first','From_Date':'first'})
        dfs.append(df_new)
    df = pd.concat(dfs)
    df = df.reindex(['Schedule','Shoot_Town','Scene_Num','Location','Actor','DATE',"D_ID","From_Date","To_Date"], axis=1)
    df.to_csv("output_pyomo.csv")
    return df

In [ ]:
scene_summary_path = '/content/drive/MyDrive/Colab Notebooks/Pyomo/Backend_Script/test/Extracted Files/Pulp_Fiction.csv'
scene_path = '/content/drive/MyDrive/Colab Notebooks/Pyomo/Backend_Script/test/Extracted Files/scene.csv'
CALENDAR_path = '/content/drive/MyDrive/Colab Notebooks/Pyomo/Backend_Script/test/Extracted Files/duration.csv'
character_path = '/content/drive/MyDrive/Colab Notebooks/Pyomo/Backend_Script/test/Extracted Files/character.csv'
location_path = '/content/drive/MyDrive/Colab Notebooks/Pyomo/Backend_Script/test/Extracted Files/location.csv'

#pyomo_df = pyomo_function(scene_summary_path, scene_path, CALENDAR_path, character_path, location_path)

In [ ]:
    # Model type
    model = ConcreteModel()

    # Input data
    scene_summary = pd.read_csv(scene_summary_path)
    scene_summary.columns = ["Shoot_Location","Scene_Number","Shoot_LocationType","Day_Part", "Actors"]
    scene = pd.read_csv(scene_path)
    scene.columns = ["Scene_Number","Shoot_Town", "Shoot_time"]
    SCENE_SUMMARY = pd.merge(scene_summary,scene,on='Scene_Number',how='left')
    SCENE_SUMMARY["Loc_Town"] = SCENE_SUMMARY[["Shoot_Town", "Shoot_Location"]].apply("_".join, axis=1)
    CALENDAR = pd.read_csv(CALENDAR_path)
    CALENDAR.columns = ["DATE","D_ID"]
    CALENDAR['DATE'] = pd.to_datetime(CALENDAR['DATE'], format='%d-%m-%Y')
    character = pd.read_csv(character_path)
    character['from_date'] = pd.to_datetime(character['from_date'], format='%d-%m-%Y')
    character['to_date'] = pd.to_datetime(character['to_date'], format='%d-%m-%Y')
    ACTOR_SUMMARY = character[["characters","criticality"]]
    ACTOR_SUMMARY.columns = ["Actors", "Criticality"]
    df = character[["characters","from_date","to_date"]]
    df['DATE'] = [pd.date_range(s, e, freq='d') for s, e in
                  zip(df['from_date'],df['to_date'])]
    df = df.explode('DATE').drop(['from_date', 'to_date'], axis=1)
    df.columns = ["Actors","DATE"]

    ACTOR_AVAILABILITY =  pd.merge(df,CALENDAR,on='DATE',how='inner')
    location = pd.read_csv(location_path)
    location['from_date'] = pd.to_datetime(location['from_date'], format='%d-%m-%Y')
    location['to_date'] = pd.to_datetime(location['to_date'], format='%d-%m-%Y')
    LOCATION_SUMMARY = location[["location","criticality"]]
    LOCATION_SUMMARY.columns = ["Shoot_Location", "Criticality"]
    LOCATION_SUMMARY = pd.merge(SCENE_SUMMARY,LOCATION_SUMMARY,on='Shoot_Location',how='inner')
    LOCATION_SUMMARY.drop(["Shoot_LocationType","Day_Part","Actors","Shoot_time"],axis=1, inplace=True)
    df1 = location[["location",	"from_date", "to_date"]]
    df1['DATE'] = [pd.date_range(s, e, freq='d') for s, e in
                  zip(df1['from_date'],df1['to_date'])]
    df1 = df1.explode('DATE').drop(['from_date', 'to_date'], axis=1)
    df1.columns = ["Shoot_Location","DATE"]
    df1 = pd.merge(SCENE_SUMMARY,df1,on='Shoot_Location',how='inner')
    df1.drop(["Shoot_LocationType","Day_Part","Actors","Shoot_time"],axis=1, inplace=True)

    LOCATION_AVAILABILITY =  pd.merge(df1,CALENDAR,on='DATE',how='inner')
    LOCATION_AVAILABILITY
    ac_sc = dict()
    for idx, val in enumerate(SCENE_SUMMARY.Actors):
        ele = SCENE_SUMMARY.Scene_Number[idx]
        if pd.notna(val):
            val = re.sub("[^A-Za-z0-9, ]+", "", val).split(', ')
            ac_sc.setdefault(ele, []).append(val)
        else:
            ac_sc.setdefault(ele, []).append("")

    character_list = character["characters"].unique().tolist()
    sdf = pd.DataFrame()
    for i in ac_sc:
        if i in range(1, len(ac_sc)+1):
            for j in range(len(ac_sc[i][0])):
                sdf = sdf.append({ac_sc[i][0][j]:i}, ignore_index=True)
    sam_dict1 = {}
    for i in character_list:
        sam_dict1[i] = sdf[i].dropna().tolist()
    sdf1 = pd.DataFrame(sam_dict1.items(), columns=['Actors', 'Scenes'])
    data1 = [(row.Actors, sample) for row in sdf1.itertuples() for sample in row.Scenes]
    ACTOR_SCENES = pd.DataFrame(data1, columns=['Actors', 'Scene_Number'])
    ACTOR_INFO = pd.merge(ACTOR_AVAILABILITY,ACTOR_SCENES,on='Actors',how='inner')
    ACTOR_INFO.rename(columns= {'D_ID':'Availability'}, inplace=True)
    ACTOR_INFO['scen_avai'] = ACTOR_INFO[['Scene_Number','Availability']].apply(tuple, axis=1)
    ACTOR_INFO['actor_avai'] = ACTOR_INFO[['Actors','Availability']].apply(tuple, axis=1)
    ACTOR_INFO['scene_actor'] = ACTOR_INFO[['Scene_Number','Actors']].apply(tuple, axis=1)
    ACTOR_INFO['asn'] = ACTOR_INFO[['Actors','Scene_Number','Availability']].apply(tuple, axis=1)
    loc_list = SCENE_SUMMARY["Loc_Town"].unique().tolist()
    loc_sc = dict(zip(SCENE_SUMMARY.Scene_Number, SCENE_SUMMARY.Loc_Town))
    sdf2 = pd.DataFrame()

    for i in loc_sc:
        if i in range(1, len(loc_sc)+1):
            sdf2 = sdf2.append({loc_sc[i]:i}, ignore_index=True)

    sam_dict2 = {}
    for i in loc_list:
        sam_dict2[i] = sdf2[i].dropna().tolist()
    sdf3 = pd.DataFrame(sam_dict2.items(), columns=['Loc', 'Scenes'])
    data2 = [(row.Loc, sample) for row in sdf3.itertuples() for sample in row.Scenes]
    LOCATION_SCENES = pd.DataFrame(data2, columns=['Loc', 'Scene_Number'])
    split = LOCATION_SCENES['Loc'].str.split('_', 1, expand=True)
    LOCATION_SCENES = LOCATION_SCENES.assign(Shoot_Town=split[0], Shoot_Location=split[1])
    LOCATION_SCENES.drop('Loc', 1, inplace=True)
    LOCATION_INFO = pd.merge(LOCATION_AVAILABILITY,LOCATION_SCENES,on=['Shoot_Location','Shoot_Town','Scene_Number'],how='inner')
    LOCATION_INFO.rename(columns= {'D_ID':'Availability'}, inplace=True)
    LOCATION_INFO['scen_avai'] = LOCATION_INFO[['Scene_Number','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['town_avai'] = LOCATION_INFO[['Shoot_Town','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['loc_avai'] = LOCATION_INFO[['Shoot_Location','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['lsn'] = LOCATION_INFO[['Shoot_Location','Scene_Number','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['tsn'] = LOCATION_INFO[['Shoot_Town','Scene_Number','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['ij'] = LOCATION_INFO[['Shoot_Town','Shoot_Location']].apply(tuple, axis=1)
    LOCATION_INFO['sj'] = LOCATION_INFO[['Scene_Number','Shoot_Location']].apply(tuple, axis=1)

    scene_list = ACTOR_INFO.Scene_Number.unique().tolist()
    scene_list.sort()
    df1 = ACTOR_INFO.groupby(by=["Scene_Number"])
    #main_df = pd.DataFrame(["scen_avail"])
    main_list = []
    for i in scene_list:
        df2 = df1.get_group(i)
        actors_list = df2.Actors.unique().tolist()
        #print("---------------")

        list = []
        df3 = df2.groupby("Actors")
        for j in actors_list:
            list.append(df3.get_group(j))

        final_df = reduce(lambda  left,right: pd.merge(left,right,on=['Availability'], how='inner'), list)
        #print(final_df.shape)

        main_list.append(final_df.iloc[:,4:5].values)

    list1 =[]
    for i in range(len(main_list)):
      #print(i)
      list1.append(main_list[i].tolist())
    flat_list = [item for sublist in list1 for item in sublist]
    flat_list1 = [item for sublist in flat_list for item in sublist]
    Actor_avail = flat_list1
    location_avail = LOCATION_INFO.scen_avai.unique().tolist()
    def Intersection(lst1, lst2):
        return set(lst2).intersection(lst1)

    fin_lis = Intersection(Actor_avail,location_avail)
    FINAL_ACTOR_INFO = ACTOR_INFO.copy()
    FINAL_ACTOR_INFO = FINAL_ACTOR_INFO[FINAL_ACTOR_INFO['scen_avai'].isin(fin_lis)]
    FINAL_LOCATION_INFO = LOCATION_INFO.copy()
    FINAL_LOCATION_INFO = FINAL_LOCATION_INFO[FINAL_LOCATION_INFO['scen_avai'].isin(fin_lis)]


In [ ]:
    # Constraints
    # Summation over n across Usn ==1
    def rule2(model, s):
        return sum(model.Usn[s,iter] for iter in model.n if (s!=None and (s,iter)) in model.Usn) == 1
    model.const1 = Constraint( model.s, rule=rule2)

AttributeError: ignored

In [ ]:
    # Model type
    model = ConcreteModel()

    # Input data
    scene_summary = pd.read_csv(scene_summary_path)
    scene_summary.columns = ["Shoot_Location","Scene_Number","Shoot_LocationType","Day_Part", "Actors"]
    scene = pd.read_csv(scene_path)
    scene.columns = ["Scene_Number","Shoot_Town", "Shoot_time"]
    SCENE_SUMMARY = pd.merge(scene_summary,scene,on='Scene_Number',how='left')
    SCENE_SUMMARY["Loc_Town"] = SCENE_SUMMARY[["Shoot_Town", "Shoot_Location"]].apply("_".join, axis=1)
    CALENDAR = pd.read_csv(CALENDAR_path)
    CALENDAR.columns = ["DATE","D_ID"]
    CALENDAR['DATE'] = pd.to_datetime(CALENDAR['DATE'], format='%d-%m-%Y')
    character = pd.read_csv(character_path)
    character['from_date'] = pd.to_datetime(character['from_date'], format='%d-%m-%Y')
    character['to_date'] = pd.to_datetime(character['to_date'], format='%d-%m-%Y')
    ACTOR_SUMMARY = character[["characters","criticality"]]
    ACTOR_SUMMARY.columns = ["Actors", "Criticality"]
    df = character[["characters","from_date","to_date"]]
    df['DATE'] = [pd.date_range(s, e, freq='d') for s, e in
                  zip(df['from_date'],df['to_date'])]
    df = df.explode('DATE').drop(['from_date', 'to_date'], axis=1)
    df.columns = ["Actors","DATE"]
    ACTOR_AVAILABILITY =  pd.merge(df,CALENDAR,on='DATE',how='inner')
    location = pd.read_csv(location_path)
    location['from_date'] = pd.to_datetime(location['from_date'], format='%d-%m-%Y')
    location['to_date'] = pd.to_datetime(location['to_date'], format='%d-%m-%Y')
    LOCATION_SUMMARY = location[["location","criticality"]]
    LOCATION_SUMMARY.columns = ["Shoot_Location", "Criticality"]
    LOCATION_SUMMARY = pd.merge(SCENE_SUMMARY,LOCATION_SUMMARY,on='Shoot_Location',how='inner')
    LOCATION_SUMMARY.drop(["Shoot_LocationType","Day_Part","Actors","Shoot_time"],axis=1, inplace=True)
    df1 = location[["location",	"from_date", "to_date"]]
    df1['DATE'] = [pd.date_range(s, e, freq='d') for s, e in
                  zip(df1['from_date'],df1['to_date'])]
    df1 = df1.explode('DATE').drop(['from_date', 'to_date'], axis=1)
    df1.columns = ["Shoot_Location","DATE"]
    df1 = pd.merge(SCENE_SUMMARY,df1,on='Shoot_Location',how='inner')
    df1.drop(["Shoot_LocationType","Day_Part","Actors","Shoot_time"],axis=1, inplace=True)

    LOCATION_AVAILABILITY =  pd.merge(df1,CALENDAR,on='DATE',how='inner')
    LOCATION_AVAILABILITY
    ac_sc = dict()
    for idx, val in enumerate(SCENE_SUMMARY.Actors):
        ele = SCENE_SUMMARY.Scene_Number[idx]
        if pd.notna(val):
            val = re.sub("[^A-Za-z0-9, ]+", "", val).split(', ')
            ac_sc.setdefault(ele, []).append(val)
        else:
            ac_sc.setdefault(ele, []).append("")

    character_list = character["characters"].unique().tolist()
    sdf = pd.DataFrame()
    for i in ac_sc:
        if i in range(1, len(ac_sc)+1):
            for j in range(len(ac_sc[i][0])):
                sdf = sdf.append({ac_sc[i][0][j]:i}, ignore_index=True)
    sam_dict1 = {}
    for i in character_list:
        sam_dict1[i] = sdf[i].dropna().tolist()
    sdf1 = pd.DataFrame(sam_dict1.items(), columns=['Actors', 'Scenes'])
    data1 = [(row.Actors, sample) for row in sdf1.itertuples() for sample in row.Scenes]
    ACTOR_SCENES = pd.DataFrame(data1, columns=['Actors', 'Scene_Number'])
    ACTOR_INFO = pd.merge(ACTOR_AVAILABILITY,ACTOR_SCENES,on='Actors',how='inner')
    ACTOR_INFO.rename(columns= {'D_ID':'Availability'}, inplace=True)
    ACTOR_INFO['scen_avai'] = ACTOR_INFO[['Scene_Number','Availability']].apply(tuple, axis=1)
    ACTOR_INFO['actor_avai'] = ACTOR_INFO[['Actors','Availability']].apply(tuple, axis=1)
    ACTOR_INFO['scene_actor'] = ACTOR_INFO[['Scene_Number','Actors']].apply(tuple, axis=1)
    ACTOR_INFO['asn'] = ACTOR_INFO[['Actors','Scene_Number','Availability']].apply(tuple, axis=1)
    loc_list = SCENE_SUMMARY["Loc_Town"].unique().tolist()
    loc_sc = dict(zip(SCENE_SUMMARY.Scene_Number, SCENE_SUMMARY.Loc_Town))
    sdf2 = pd.DataFrame()

    for i in loc_sc:
        if i in range(1, len(loc_sc)+1):
            sdf2 = sdf2.append({loc_sc[i]:i}, ignore_index=True)

    sam_dict2 = {}
    for i in loc_list:
        sam_dict2[i] = sdf2[i].dropna().tolist()
    sdf3 = pd.DataFrame(sam_dict2.items(), columns=['Loc', 'Scenes'])
    data2 = [(row.Loc, sample) for row in sdf3.itertuples() for sample in row.Scenes]
    LOCATION_SCENES = pd.DataFrame(data2, columns=['Loc', 'Scene_Number'])
    split = LOCATION_SCENES['Loc'].str.split('_', 1, expand=True)
    LOCATION_SCENES = LOCATION_SCENES.assign(Shoot_Town=split[0], Shoot_Location=split[1])
    LOCATION_SCENES.drop('Loc', 1, inplace=True)
    LOCATION_INFO = pd.merge(LOCATION_AVAILABILITY,LOCATION_SCENES,on=['Shoot_Location','Shoot_Town','Scene_Number'],how='inner')
    LOCATION_INFO.rename(columns= {'D_ID':'Availability'}, inplace=True)
    LOCATION_INFO['scen_avai'] = LOCATION_INFO[['Scene_Number','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['town_avai'] = LOCATION_INFO[['Shoot_Town','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['loc_avai'] = LOCATION_INFO[['Shoot_Location','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['lsn'] = LOCATION_INFO[['Shoot_Location','Scene_Number','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['tsn'] = LOCATION_INFO[['Shoot_Town','Scene_Number','Availability']].apply(tuple, axis=1)
    LOCATION_INFO['ij'] = LOCATION_INFO[['Shoot_Town','Shoot_Location']].apply(tuple, axis=1)
    LOCATION_INFO['sj'] = LOCATION_INFO[['Scene_Number','Shoot_Location']].apply(tuple, axis=1)

    scene_list = ACTOR_INFO.Scene_Number.unique().tolist()
    scene_list.sort()
    df1 = ACTOR_INFO.groupby(by=["Scene_Number"])
    #main_df = pd.DataFrame(["scen_avail"])
    main_list = []
    for i in scene_list:
        df2 = df1.get_group(i)
        actors_list = df2.Actors.unique().tolist()
        #print("---------------")

        list = []
        df3 = df2.groupby("Actors")
        for j in actors_list:
            list.append(df3.get_group(j))

        final_df = reduce(lambda  left,right: pd.merge(left,right,on=['Availability'], how='inner'), list)
        #print(final_df.shape)

        main_list.append(final_df.iloc[:,4:5].values)

    list1 =[]
    for i in range(len(main_list)):
      #print(i)
      list1.append(main_list[i].tolist())
    flat_list = [item for sublist in list1 for item in sublist]
    flat_list1 = [item for sublist in flat_list for item in sublist]
    Actor_avail = flat_list1
    location_avail = LOCATION_INFO.scen_avai.unique().tolist()
    def Intersection(lst1, lst2):
        return set(lst2).intersection(lst1)

    fin_lis = Intersection(Actor_avail,location_avail)
    FINAL_ACTOR_INFO = ACTOR_INFO.copy()
    FINAL_ACTOR_INFO = FINAL_ACTOR_INFO[FINAL_ACTOR_INFO['scen_avai'].isin(fin_lis)]
    FINAL_LOCATION_INFO = LOCATION_INFO.copy()
    FINAL_LOCATION_INFO = FINAL_LOCATION_INFO[FINAL_LOCATION_INFO['scen_avai'].isin(fin_lis)]

    # sets
    model.i = Set(initialize=LOCATION_SUMMARY["Shoot_Town"].tolist())
    model.j = Set(initialize=LOCATION_SUMMARY["Shoot_Location"].tolist())
    model.s = Set(initialize=SCENE_SUMMARY["Scene_Number"].tolist())
    model.a = Set(initialize=ACTOR_SUMMARY["Actors"].tolist())
    model.n = Set(initialize=CALENDAR['D_ID'].tolist())
    model.an = Set(initialize=FINAL_ACTOR_INFO["actor_avai"].tolist())
    model.sa = Set(initialize=FINAL_ACTOR_INFO["scene_actor"].tolist())
    model.asn = Set(initialize=FINAL_ACTOR_INFO["asn"].tolist())
    model.jn = Set(initialize=FINAL_LOCATION_INFO["loc_avai"].tolist())
    model.jsn = Set(initialize=FINAL_LOCATION_INFO["lsn"].tolist())
    model.tn = Set(initialize=FINAL_LOCATION_INFO["town_avai"].tolist())
    model.isn = Set(initialize=FINAL_LOCATION_INFO["tsn"].tolist())
    model.ij = Set(initialize=FINAL_LOCATION_INFO["ij"].tolist())
    model.sj = Set(initialize=FINAL_LOCATION_INFO["sj"].tolist())

    # parameters
    set_ele1 = pd.Series(ACTOR_SUMMARY.Criticality.values,index=ACTOR_SUMMARY.Actors).to_dict()
    model.CA = Param(model.a,initialize=set_ele1)
    set_elem = pd.Series(SCENE_SUMMARY.Shoot_time.values,index=SCENE_SUMMARY.Scene_Number).to_dict()
    model.TS = Param(model.s,initialize=set_elem)
    model.Can = Param(model.an, initialize=1)
    set_elex = pd.Series(LOCATION_SUMMARY.Criticality.values,index=LOCATION_SUMMARY.Shoot_Location).to_dict()
    model.CL = Param(model.j,initialize=set_elex)
    model.Bsa = Param(model.sa, initialize=1)
    model.Can = Param(model.an, initialize=1)
    model.Asj = Param(model.sj, initialize=1)
    model.Djn = Param(model.jn, initialize=1)
    model.Eij = Param(model.ij, initialize=1)
    set_ele4 = pd.Series(SCENE_SUMMARY.Shoot_time.values,index=SCENE_SUMMARY.Scene_Number).to_dict()
    model.Tsh = Param(model.s ,initialize=set_ele4)

    # variables
    model.Wjn = Var(model.jn, within=Binary)
    model.Dursn = Var( model.s,model.n)
    model.AVsn = Var( model.s,model.n)
    model.SHsn = Var( model.s,model.n)
    model.DELsn =Var(model.s,model.n, within=Binary)
    model.Usn = Var(fin_lis,within=Binary)
    model.Vasn = Var(model.asn, within=Binary)
    model.Win = Var(model.tn,within=Binary)
    model.Xjsn = Var( model.jsn ,within=Binary)

    # Constraints
    # Summation over n across Usn ==1
    def rule2(model, s):
        return sum(model.Usn[s,iter] for iter in model.n if (s!=None and (s,iter)) in model.Usn) == 1
    model.const1 = Constraint( model.s, rule=rule2)

This is usually indicative of a modelling error.
To avoid this warning, use block.del_component() and block.add_component().
ERROR:pyomo.core:Rule failed when generating expression for Constraint const1 with index 15:
ValueError: Invalid constraint expression. The constraint expression resolved to a trivial Boolean (False) instead of a Pyomo object. Please modify your rule to return Constraint.Infeasible instead of False.

Error thrown for Constraint 'const1[15]'
ERROR:pyomo.core:Constructing component 'const1' from data=None failed:
ValueError: Invalid constraint expression. The constraint expression resolved to a trivial Boolean (False) instead of a Pyomo object. Please modify your rule to return Constraint.Infeasible instead of False.

Error thrown for Constraint 'const1[15]'


ValueError: ignored

In [ ]:
 Actor_avail

[(2.0, 4.0),
 (2.0, 5.0),
 (2.0, 6.0),
 (2.0, 7.0),
 (2.0, 8.0),
 (2.0, 9.0),
 (2.0, 10.0),
 (2.0, 11.0),
 (2.0, 12.0),
 (2.0, 13.0),
 (2.0, 14.0),
 (2.0, 15.0),
 (3.0, 2.0),
 (3.0, 3.0),
 (3.0, 4.0),
 (3.0, 5.0),
 (3.0, 6.0),
 (3.0, 7.0),
 (3.0, 8.0),
 (3.0, 9.0),
 (3.0, 10.0),
 (3.0, 11.0),
 (3.0, 12.0),
 (3.0, 13.0),
 (3.0, 14.0),
 (3.0, 15.0),
 (3.0, 16.0),
 (3.0, 17.0),
 (3.0, 18.0),
 (3.0, 19.0),
 (3.0, 20.0),
 (3.0, 21.0),
 (3.0, 22.0),
 (3.0, 23.0),
 (3.0, 24.0),
 (3.0, 25.0),
 (3.0, 26.0),
 (3.0, 27.0),
 (3.0, 28.0),
 (3.0, 29.0),
 (3.0, 32.0),
 (3.0, 33.0),
 (3.0, 34.0),
 (3.0, 35.0),
 (3.0, 36.0),
 (3.0, 37.0),
 (3.0, 38.0),
 (3.0, 39.0),
 (3.0, 40.0),
 (3.0, 41.0),
 (3.0, 42.0),
 (3.0, 43.0),
 (3.0, 44.0),
 (3.0, 45.0),
 (3.0, 46.0),
 (3.0, 47.0),
 (3.0, 48.0),
 (3.0, 49.0),
 (3.0, 50.0),
 (3.0, 51.0),
 (3.0, 52.0),
 (3.0, 53.0),
 (3.0, 54.0),
 (3.0, 55.0),
 (3.0, 56.0),
 (3.0, 57.0),
 (3.0, 58.0),
 (3.0, 59.0),
 (4.0, 2.0),
 (4.0, 3.0),
 (4.0, 4.0),
 (4.0, 5.0),
 (4.0, 6.0

In [ ]:
model.Usn.pprint()

Usn : Size=297, Index=Usn_index
    Key          : Lower : Value : Upper : Fixed : Stale : Domain
      (2.0, 4.0) :     0 :  None :     1 : False :  True : Binary
      (2.0, 5.0) :     0 :  None :     1 : False :  True : Binary
      (2.0, 6.0) :     0 :  None :     1 : False :  True : Binary
      (2.0, 7.0) :     0 :  None :     1 : False :  True : Binary
      (2.0, 8.0) :     0 :  None :     1 : False :  True : Binary
      (2.0, 9.0) :     0 :  None :     1 : False :  True : Binary
     (2.0, 10.0) :     0 :  None :     1 : False :  True : Binary
     (2.0, 11.0) :     0 :  None :     1 : False :  True : Binary
     (2.0, 12.0) :     0 :  None :     1 : False :  True : Binary
     (2.0, 13.0) :     0 :  None :     1 : False :  True : Binary
     (2.0, 14.0) :     0 :  None :     1 : False :  True : Binary
     (2.0, 15.0) :     0 :  None :     1 : False :  True : Binary
     (3.0, 10.0) :     0 :  None :     1 : False :  True : Binary
     (3.0, 11.0) :     0 :  None :     1 : F

In [ ]:
model.const1.pprint()
model.Usn[s,iter] for iter in model.n if (s!=None and (s,iter)) in model.Usn) == 1

SyntaxError: ignored

In [ ]:
for i in model.s:
  for j in model.n:
    if i!=None and (i,j):
      print(i,j)
    #print(model.Usn[i,j])
    #sdf = sdf.append({ac_sc[i][0][j]:i}, ignore_index=True)




2 1
2 2
2 3
2 4
2 5
2 6
2 7
2 8
2 9
2 10
2 11
2 12
2 13
2 14
2 15
2 16
2 17
2 18
2 19
2 20
2 21
2 22
2 23
2 24
2 25
2 26
2 27
2 28
2 29
2 30
2 31
2 32
2 33
2 34
2 35
2 36
2 37
2 38
2 39
2 40
2 41
2 42
2 43
2 44
2 45
2 46
2 47
2 48
2 49
2 50
2 51
2 52
2 53
2 54
2 55
2 56
2 57
2 58
2 59
2 60
2 61
2 62
2 63
2 64
2 65
2 66
2 67
2 68
2 69
2 70
2 71
2 72
2 73
2 74
2 75
2 76
2 77
2 78
2 79
2 80
2 81
2 82
2 83
2 84
2 85
2 86
2 87
2 88
2 89
2 90
3 1
3 2
3 3
3 4
3 5
3 6
3 7
3 8
3 9
3 10
3 11
3 12
3 13
3 14
3 15
3 16
3 17
3 18
3 19
3 20
3 21
3 22
3 23
3 24
3 25
3 26
3 27
3 28
3 29
3 30
3 31
3 32
3 33
3 34
3 35
3 36
3 37
3 38
3 39
3 40
3 41
3 42
3 43
3 44
3 45
3 46
3 47
3 48
3 49
3 50
3 51
3 52
3 53
3 54
3 55
3 56
3 57
3 58
3 59
3 60
3 61
3 62
3 63
3 64
3 65
3 66
3 67
3 68
3 69
3 70
3 71
3 72
3 73
3 74
3 75
3 76
3 77
3 78
3 79
3 80
3 81
3 82
3 83
3 84
3 85
3 86
3 87
3 88
3 89
3 90
4 1
4 2
4 3
4 4
4 5
4 6
4 7
4 8
4 9
4 10
4 11
4 12
4 13
4 14
4 15
4 16
4 17
4 18
4 19
4 20
4 21
4 22
4 23
4 24
4 25
4 

In [ ]:
ac_sc

{2: [['Amala', 'Devi', 'Ganesh', 'Madan', 'Jyothi']],
 3: [['Vijay', 'Chai']],
 4: [['Vijay', 'Chai']],
 7: [['Vijay', 'Chai', 'Amala', 'Girl1']],
 8: [['Devi', 'Arnav']],
 9: [['Vijay', 'Chai']],
 10: [['Amala', 'Ganesh']],
 11: [['Vijay', ' Amala']],
 12: [['Amala']],
 13: [['Vijay', 'Chai', 'Rani', 'Sruti']],
 15: [['Devi', 'Madan']],
 16: [['Vijay', 'Arnav']],
 17: [['Vijay', 'Ganesh', 'Jyothi']]}

In [ ]:
FINAL_LOCATION_INFO

,Shoot_Location,Scene_Number,Shoot_Town,Loc_Town,DATE,Availability,scen_avai,town_avai,loc_avai,lsn,tsn,ij,sj
11,Dwaraka,3,Delhi,Delhi_Dwaraka,2023-01-10,10,"(3, 10)","(Delhi, 10)","(Dwaraka, 10)","(Dwaraka, 3, 10)","(Delhi, 3, 10)","(Delhi, Dwaraka)","(3, Dwaraka)"
12,Dwaraka,3,Delhi,Delhi_Dwaraka,2023-01-11,11,"(3, 11)","(Delhi, 11)","(Dwaraka, 11)","(Dwaraka, 3, 11)","(Delhi, 3, 11)","(Delhi, Dwaraka)","(3, Dwaraka)"
13,Dwaraka,3,Delhi,Delhi_Dwaraka,2023-01-12,12,"(3, 12)","(Delhi, 12)","(Dwaraka, 12)","(Dwaraka, 3, 12)","(Delhi, 3, 12)","(Delhi, Dwaraka)","(3, Dwaraka)"
14,Dwaraka,3,Delhi,Delhi_Dwaraka,2023-01-13,13,"(3, 13)","(Delhi, 13)","(Dwaraka, 13)","(Dwaraka, 3, 13)","(Delhi, 3, 13)","(Delhi, Dwaraka)","(3, Dwaraka)"
15,Dwaraka,3,Delhi,Delhi_Dwaraka,2023-01-14,14,"(3, 14)","(Delhi, 14)","(Dwaraka, 14)","(Dwaraka, 3, 14)","(Delhi, 3, 14)","(Delhi, Dwaraka)","(3, Dwaraka)"
16,Dwaraka,3,Delhi,Delhi_Dwaraka,2023-01-15,15,"(3, 15)","(Delhi, 15)","(Dwaraka, 15)","(Dwaraka, 3, 15)","(Delhi, 3, 15)","(Delhi, Dwaraka)","(3, Dwaraka)"
17,Dwaraka,3,Delhi,Delhi_Dwaraka,2023-01-16,16,"(3, 16)","(Delhi, 16)","(Dwaraka, 16)","(Dwaraka, 3, 16)","(Delhi, 3, 16)","(Delhi, Dwaraka)","(3, Dwaraka)"
18,Dwaraka,3,Delhi,Delhi_Dwaraka,2023-01-17,17,"(3, 17)","(Delhi, 17)","(Dwaraka, 17)","(Dwaraka, 3, 17)","(Delhi, 3, 17)","(Delhi, Dwaraka)","(3, Dwaraka)"
19,Dwaraka,3,Delhi,Delhi_Dwaraka,2023-01-18,18,"(3, 18)","(Delhi, 18)","(Dwaraka, 18)","(Dwaraka, 3, 18)","(Delhi, 3, 18)","(Delhi, Dwaraka)","(3, Dwaraka)"
20,Dwaraka,3,Delhi,Delhi_Dwaraka,2023-01-19,19,"(3, 19)","(Delhi, 19)","(Dwaraka, 19)","(Dwaraka, 3, 19)","(Delhi, 3, 19)","(Delhi, Dwaraka)","(3, Dwaraka)"


In [ ]:
ac_sc

{1: [''],
 2: [['Amala', 'Devi', 'Ganesh', 'Madan', 'Jyothi']],
 3: [['Vijay', 'Chai']],
 4: [['Vijay', 'Chai']],
 5: [''],
 6: [''],
 7: [['Vijay', 'Chai', 'Amala', 'Girl1']],
 8: [['Devi', 'Arnav']],
 9: [['Vijay', 'Chai']],
 10: [['Amala', 'Ganesh']],
 11: [['Vijay', ' Amala']],
 12: [['Amala']],
 13: [['Vijay', 'Chai', 'Rani', 'Sruti']],
 14: [''],
 15: [['Devi', 'Madan']],
 16: [['Vijay', 'Arnav']],
 17: [['Vijay', 'Ganesh', 'Jyothi']]}